# 14 — Cross-Dataset Replication Comparison

This notebook compares the benchmark results across all available EEG/ERP datasets in the thesis pipeline.

## Purpose

The original single-dataset result (ds003061) had to be replicated on at least two additional OpenNeuro EEG/ERP datasets with similar oddball tasks.

This notebook addresses that requirement by collecting the saved benchmark outputs across datasets and producing a cross-dataset replication summary.

## What this notebook does

For each dataset family, the notebook reads:

- the main grouped within-subject benchmark
- the permuted-label control
- the pre-stimulus-only control
- the LOSO benchmark, if available

It then compares:

- LDA performance across datasets
- whether LDA remains strong or top-ranked
- whether the main benchmark beats the controls
- whether results remain above chance
- whether enough additional datasets have been completed to satisfy the replication request

## Important

This notebook does **not** train models.
It assumes the benchmark notebooks have already been run for each dataset.

Expected saved files include:

- `model_summary_across_subjects.csv`
- `subject_summary_across_seeds.csv`
- `loso_model_summary_across_subjects.csv`
- `loso_subject_summary_across_seeds.csv`

In [ ]:
# Run from the repository root so that relative result paths resolve
# identically regardless of where this notebook lives.
from oddball_eeg.paths import use_project_root

PROJECT_ROOT = use_project_root()

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from oddball_eeg.config import get_experiment
from oddball_eeg.io import save_dataframe, save_json
from oddball_eeg.visualization import (
    round_numeric_columns,
    save_figure,
    set_pandas_display_options,
)


set_pandas_display_options()

plt.rcParams["figure.dpi"] = 140
plt.rcParams["savefig.dpi"] = 300
plt.rcParams["font.size"] = 11
plt.rcParams["axes.titlesize"] = 12
plt.rcParams["axes.labelsize"] = 11
plt.rcParams["legend.fontsize"] = 10


OUT_DIR = PROJECT_ROOT / "results" / "cross_dataset_replication"
FIG_DIR = OUT_DIR / "figures"
TABLE_DIR = OUT_DIR / "tables"

FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("OUT_DIR     :", OUT_DIR)
print("FIG_DIR     :", FIG_DIR)
print("TABLE_DIR   :", TABLE_DIR)

In [ ]:
# ------------------------------------------------------------
# Main model of the LDA dissertation version
# ------------------------------------------------------------
PRIMARY_MODEL = "lda"

# Models retained in the thesis comparison
MODEL_ORDER = ["lda", "logreg", "eegnet"]

MODEL_LABELS = {
    "lda": "LDA",
    "logreg": "Logistic Regression",
    "eegnet": "EEGNet",
}

# ------------------------------------------------------------
# Replication requirement
# ------------------------------------------------------------
REFERENCE_DATASET = "ds003061"
REQUIRED_ADDITIONAL_DATASETS = 2

# ------------------------------------------------------------
# FINAL MAX15 EXPERIMENT GROUPS
# ------------------------------------------------------------
# Use only max15 keys here. Do not mix *_main or *_n15
# in the final cross-dataset tables.
# ------------------------------------------------------------

MAIN_EXPERIMENT_KEYS = [
    "ds003061_within_subject_max30_wide_timebin",
    "ds005863_within_subject_max30_wide_timebin",
    "ds006466_within_subject_max30_wide_timebin",
    "ds006480_within_subject_max30_wide_timebin",
]

TIMEBIN_EXPERIMENT_KEYS = [
    # "ds003061_within_subject_max15_timebin",
    # "ds005863_within_subject_max15_timebin",
    # "ds006466_within_subject_max15_timebin",
    # "ds006480_within_subject_max15_timebin",
]

WIDE_TIMEBIN_EXPERIMENT_KEYS = [
    # "ds003061_within_subject_max15_wide_timebin",
    # "ds005863_within_subject_max15_wide_timebin",
    # "ds006466_within_subject_max15_wide_timebin",
    # "ds006480_within_subject_max15_wide_timebin",
]

DS006480_CONDITION_EXPERIMENT_KEYS = [
    # "ds006480_active_within_subject_max15",
    # "ds006480_passive_within_subject_max15",
    # "ds006480_active_within_subject_max15_timebin",
    # "ds006480_passive_within_subject_max15_timebin",
]

# Keep these empty unless you actually ran them.
DS006480_EXTRA_CONDITION_EXPERIMENT_KEYS = [
    # "ds006480_control_within_subject_max15",
    # "ds006480_shock_within_subject_max15",
    # "ds006480_control_within_subject_max15_timebin",
    # "ds006480_shock_within_subject_max15_timebin",
]

# ------------------------------------------------------------
# Conservative thresholds for replication-status labeling
# ------------------------------------------------------------
CHANCE_AUC = 0.50
MIN_MAIN_AUC_FOR_SUCCESS = 0.60
MIN_MAIN_MINUS_PERMUTED_DELTA = 0.05
MIN_MAIN_MINUS_PRESTIM_DELTA = 0.03
TOP_K_ACCEPTED = 2

print("Primary model:", PRIMARY_MODEL)
print("Reference dataset:", REFERENCE_DATASET)
print("Required additional datasets:", REQUIRED_ADDITIONAL_DATASETS)

print("\nMain pooled experiments:")
for k in MAIN_EXPERIMENT_KEYS:
    print(" -", k)

print("\nTime-binned experiments:")
for k in TIMEBIN_EXPERIMENT_KEYS:
    print(" -", k)

print("\nWide-window time-binned experiments:")
for k in WIDE_TIMEBIN_EXPERIMENT_KEYS:
    print(" -", k)

print("\nds006480 condition-separated experiments:")
for k in DS006480_CONDITION_EXPERIMENT_KEYS:
    print(" -", k)

In [ ]:
def normalize_model_name(x: str) -> str:
    s = str(x).strip().lower()
    s = s.replace("-", "_").replace(" ", "_")

    if s in {"lda"}:
        return "lda"

    if s in {"logreg", "logistic_regression", "logisticregression", "lr", "logistic"}:
        return "logreg"

    if s in {"eegnet", "eeg_net"}:
        return "eegnet"

    return s


def model_display_name(x: str) -> str:
    return MODEL_LABELS.get(normalize_model_name(x), str(x))


def normalize_subject_column(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    if "subject" in out.columns:
        out["subject"] = (
            out["subject"]
            .astype(str)
            .str.extract(r"(\d+)")[0]
            .str.zfill(3)
        )

    return out


def normalize_model_column(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    if "model" in out.columns:
        out["model"] = out["model"].map(normalize_model_name)

    return out


def safe_float(x):
    try:
        return float(x)
    except Exception:
        return np.nan


def file_status(path: Path) -> str:
    return "found" if Path(path).exists() else "missing"


def find_result_file(cfg, filename: str) -> Path | None:
    """
    Robustly find a saved result file under the current structured results folder.

    This supports both older flat outputs like:
        cfg.result_dir() / filename

    and newer cleaner outputs like:
        cfg.result_dir() / tables / ... / filename
    """
    result_dir = cfg.result_dir()

    candidate_paths = [
        result_dir / filename,
        result_dir / "tables" / filename,
        result_dir / "tables" / "benchmark" / filename,
        result_dir / "tables" / "summaries" / filename,
        result_dir / "tables" / "results" / filename,
        result_dir / "tables" / "controls" / filename,
        result_dir / "tables" / "loso" / filename,
        result_dir / "artifacts" / filename,
    ]

    for path in candidate_paths:
        if path.exists():
            return path

    # Last-resort recursive search inside this experiment result directory
    matches = sorted(result_dir.rglob(filename))

    if len(matches) > 0:
        return matches[0]

    return None


def safe_read_result_csv(cfg, filename: str, required: bool = False) -> pd.DataFrame | None:
    path = find_result_file(cfg, filename)

    if path is None:
        if required:
            raise FileNotFoundError(
                f"Could not find {filename} under:\n{cfg.result_dir()}"
            )
        return None

    df = pd.read_csv(path)
    df = normalize_subject_column(df)
    df = normalize_model_column(df)

    # Store source path for debugging
    df.attrs["source_path"] = str(path)

    return df


def get_source_path(df: pd.DataFrame | None) -> str:
    if df is None:
        return "missing"
    return df.attrs.get("source_path", "loaded_path_unknown")

In [ ]:
def get_experiment_registry() -> dict:
    """
    Return config.EXPERIMENTS if available.
    """
    if not hasattr(config, "EXPERIMENTS"):
        raise AttributeError(
            "config.EXPERIMENTS was not found. "
            "Add an EXPERIMENTS registry to config.py."
        )

    return dict(config.EXPERIMENTS)


def get_family_keys(main_key: str) -> dict:
    """
    Return the related main/permuted/prestim/LOSO experiment keys
    for a within-subject experiment key.

    Supports both old keys such as:
        ds003061_within_subject_main

    and newer keys such as:
        ds003061_within_subject_max15
        ds003061_within_subject_max30_wide_timebin
    """

    marker = "_within_subject_"

    if marker not in main_key:
        raise ValueError(
            "Unsupported main experiment key. Expected a key containing "
            f"{marker!r}. Got: {main_key!r}"
        )

    prefix, suffix = main_key.split(marker, maxsplit=1)

    # Old original family used different control naming.
    if suffix == "main":
        return {
            "main": f"{prefix}_within_subject_main",
            "permuted": f"{prefix}_control_permuted",
            "prestim": f"{prefix}_control_prestim",
            "loso": f"{prefix}_loso_main",
        }

    # New registered families:
    # max15, n15, max15_timebin, max15_wide_timebin,
    # max30_wide_timebin, etc.
    return {
        "main": f"{prefix}_within_subject_{suffix}",
        "permuted": f"{prefix}_control_permuted_{suffix}",
        "prestim": f"{prefix}_control_prestim_{suffix}",
        "loso": f"{prefix}_loso_{suffix}",
    }


main_experiment_keys = list(MAIN_EXPERIMENT_KEYS)

if len(main_experiment_keys) == 0:
    raise RuntimeError("MAIN_EXPERIMENT_KEYS is empty.")

print("Final main experiment keys:")
for key in main_experiment_keys:
    print(" -", key)

In [ ]:
family_rows = []

for main_key in main_experiment_keys:
    family_keys = get_family_keys(main_key)

    cfg_main = get_experiment(family_keys["main"])
    cfg_perm = get_experiment(family_keys["permuted"])
    cfg_prestim = get_experiment(family_keys["prestim"])
    cfg_loso = get_experiment(family_keys["loso"])

    dataset = cfg_main.dataset.dataset
    task = cfg_main.dataset.task

    expected_files = {
        "main_model": (cfg_main, "model_summary_across_subjects.csv"),
        "main_subject": (cfg_main, "subject_summary_across_seeds.csv"),
        "perm_model": (cfg_perm, "model_summary_across_subjects.csv"),
        "prestim_model": (cfg_prestim, "model_summary_across_subjects.csv"),
        "loso_model": (cfg_loso, "loso_model_summary_across_subjects.csv"),
        "loso_subject": (cfg_loso, "loso_subject_summary_across_seeds.csv"),
    }

    row = {
        "main_experiment_key": main_key,
        "dataset": dataset,
        "task": task,
        "signal": cfg_main.signal.slug() if hasattr(cfg_main, "signal") else "",
        "main_dir": str(cfg_main.result_dir()),
        "permuted_dir": str(cfg_perm.result_dir()),
        "prestim_dir": str(cfg_prestim.result_dir()),
        "loso_dir": str(cfg_loso.result_dir()),
    }

    for label, (cfg_obj, filename) in expected_files.items():
        found_path = find_result_file(cfg_obj, filename)

        row[f"{label}_filename"] = filename
        row[f"{label}_status"] = "found" if found_path is not None else "missing"
        row[f"{label}_path"] = str(found_path) if found_path is not None else "missing"

    family_rows.append(row)

family_status_df = (
    pd.DataFrame(family_rows)
    .sort_values("dataset")
    .reset_index(drop=True)
)

save_dataframe(
    family_status_df,
    TABLE_DIR / "dataset_family_file_status.csv",
)

display(family_status_df)

In [ ]:
loaded = {}

for main_key in main_experiment_keys:
    family_keys = get_family_keys(main_key)

    cfg_main = get_experiment(family_keys["main"])
    cfg_perm = get_experiment(family_keys["permuted"])
    cfg_prestim = get_experiment(family_keys["prestim"])
    cfg_loso = get_experiment(family_keys["loso"])

    dataset = cfg_main.dataset.dataset

    loaded[dataset] = {
        "main_key": main_key,
        "task": cfg_main.dataset.task,
        "signal": cfg_main.signal.slug() if hasattr(cfg_main, "signal") else "",

        "main_model": safe_read_result_csv(cfg_main, "model_summary_across_subjects.csv"),
        "main_subject": safe_read_result_csv(cfg_main, "subject_summary_across_seeds.csv"),

        "perm_model": safe_read_result_csv(cfg_perm, "model_summary_across_subjects.csv"),
        "prestim_model": safe_read_result_csv(cfg_prestim, "model_summary_across_subjects.csv"),

        "loso_model": safe_read_result_csv(cfg_loso, "loso_model_summary_across_subjects.csv"),
        "loso_subject": safe_read_result_csv(cfg_loso, "loso_subject_summary_across_seeds.csv"),
    }

summary_load_rows = []

for dataset, obj in loaded.items():
    summary_load_rows.append({
        "dataset": dataset,
        "task": obj["task"],
        "signal": obj["signal"],

        "main_model_loaded": obj["main_model"] is not None,
        "main_model_path": get_source_path(obj["main_model"]),

        "main_subject_loaded": obj["main_subject"] is not None,
        "main_subject_path": get_source_path(obj["main_subject"]),

        "perm_model_loaded": obj["perm_model"] is not None,
        "perm_model_path": get_source_path(obj["perm_model"]),

        "prestim_model_loaded": obj["prestim_model"] is not None,
        "prestim_model_path": get_source_path(obj["prestim_model"]),

        "loso_model_loaded": obj["loso_model"] is not None,
        "loso_model_path": get_source_path(obj["loso_model"]),

        "loso_subject_loaded": obj["loso_subject"] is not None,
        "loso_subject_path": get_source_path(obj["loso_subject"]),
    })

load_status_df = (
    pd.DataFrame(summary_load_rows)
    .sort_values("dataset")
    .reset_index(drop=True)
)

save_dataframe(
    load_status_df,
    TABLE_DIR / "dataset_load_status.csv",
)

display(load_status_df)

In [ ]:
model_rows = []

for dataset, obj in loaded.items():
    df = obj["main_model"]

    if df is None:
        print(f"[MISSING] {dataset}: no main model summary found.")
        print("Expected somewhere under the result directory for:")
        print(" ", obj["main_key"])
        continue

    required_cols = {"model", "mean_subject_auc"}

    if not required_cols.issubset(df.columns):
        print(f"[WARN] Skipping {dataset}: main model summary missing required columns.")
        print("Available columns:", df.columns.tolist())
        print("Source path:", get_source_path(df))
        continue

    print(f"[OK] Loaded main model summary for {dataset}")
    print("     Source:", get_source_path(df))
    print("     Shape :", df.shape)

    for _, r in df.iterrows():
        model = normalize_model_name(r["model"])

        model_rows.append({
            "dataset": dataset,
            "task": obj["task"],
            "signal": obj["signal"],
            "model": model,
            "model_display": model_display_name(model),
            "main_mean_auc": safe_float(r.get("mean_subject_auc")),
            "main_std_auc": safe_float(r.get("std_subject_auc")),
            "main_median_auc": safe_float(r.get("median_subject_auc")),
            "main_worst_auc": safe_float(r.get("worst_subject_auc")),
            "main_best_auc": safe_float(r.get("best_subject_auc")),
            "main_mean_pr_auc": safe_float(r.get("mean_subject_pr_auc")),
            "main_mean_bal_acc": safe_float(r.get("mean_subject_bal_acc")),
            "main_mean_acc": safe_float(r.get("mean_subject_acc")),
            "main_mean_f1": safe_float(r.get("mean_subject_f1")),
        })

cross_model_df = pd.DataFrame(model_rows)

if len(cross_model_df) == 0:
    print("\nNo main model summaries were loaded.")
    print("Check the table below to see which files are missing:")
    display(load_status_df)

    raise RuntimeError(
        "No main model summaries were loaded. "
        "Either Notebook 02 has not been run for these datasets, "
        "or the summary files are saved under a filename/path not searched yet."
    )

cross_model_df = (
    cross_model_df
    .sort_values(["dataset", "main_mean_auc"], ascending=[True, False])
    .reset_index(drop=True)
)

save_dataframe(
    cross_model_df,
    TABLE_DIR / "cross_dataset_model_summary.csv",
)

display(round_numeric_columns(cross_model_df, 4))

In [ ]:
def load_model_summary_for_experiment(exp_key: str, group_name: str) -> pd.DataFrame:
    """
    Load model_summary_across_subjects.csv for one standalone experiment key.
    """
    cfg = get_experiment(exp_key)
    df = safe_read_result_csv(cfg, "model_summary_across_subjects.csv")

    if df is None:
        print(f"[MISSING] {group_name}: {exp_key}")
        print("Expected under:", cfg.result_dir())
        return pd.DataFrame()

    if "model" not in df.columns or "mean_subject_auc" not in df.columns:
        print(f"[WARN] {group_name}: {exp_key} missing required columns.")
        print("Columns:", df.columns.tolist())
        print("Source :", get_source_path(df))
        return pd.DataFrame()

    out_rows = []

    for _, r in df.iterrows():
        model = normalize_model_name(r["model"])

        out_rows.append({
            "group": group_name,
            "experiment_key": exp_key,
            "dataset": cfg.dataset.dataset,
            "task": cfg.dataset.task,
            "event_preset": cfg.dataset.event_preset,
            "run_tag": cfg.run_tag,
            "crop_window": str(cfg.effective_crop_window()),
            "feature_mode": cfg.feature.mode,
            "feature_bin_ms": cfg.feature.bin_ms,
            "model": model,
            "model_display": model_display_name(model),
            "mean_subject_auc": safe_float(r.get("mean_subject_auc")),
            "std_subject_auc": safe_float(r.get("std_subject_auc")),
            "median_subject_auc": safe_float(r.get("median_subject_auc")),
            "worst_subject_auc": safe_float(r.get("worst_subject_auc")),
            "best_subject_auc": safe_float(r.get("best_subject_auc")),
            "mean_subject_pr_auc": safe_float(r.get("mean_subject_pr_auc")),
            "mean_subject_bal_acc": safe_float(r.get("mean_subject_bal_acc")),
            "mean_subject_acc": safe_float(r.get("mean_subject_acc")),
            "mean_subject_f1": safe_float(r.get("mean_subject_f1")),
            "n_subjects": safe_float(r.get("n_subjects")),
            "source_path": get_source_path(df),
        })

    return pd.DataFrame(out_rows)


refinement_frames = []

for key in TIMEBIN_EXPERIMENT_KEYS:
    refinement_frames.append(load_model_summary_for_experiment(key, "timebin_0_0p6"))

for key in WIDE_TIMEBIN_EXPERIMENT_KEYS:
    refinement_frames.append(load_model_summary_for_experiment(key, "wide_timebin_0_0p8"))

for key in DS006480_CONDITION_EXPERIMENT_KEYS:
    refinement_frames.append(load_model_summary_for_experiment(key, "ds006480_condition"))

for key in DS006480_EXTRA_CONDITION_EXPERIMENT_KEYS:
    refinement_frames.append(load_model_summary_for_experiment(key, "ds006480_extra_condition"))

refinement_frames = [df for df in refinement_frames if df is not None and len(df) > 0]

if len(refinement_frames) == 0:
    print("[WARN] No refinement summaries loaded.")
    refinement_summary_df = pd.DataFrame()
else:
    refinement_summary_df = (
        pd.concat(refinement_frames, ignore_index=True)
        .sort_values(["group", "dataset", "event_preset", "model"])
        .reset_index(drop=True)
    )

save_dataframe(
    refinement_summary_df,
    TABLE_DIR / "refinement_model_summary_all.csv",
)

display(round_numeric_columns(refinement_summary_df, 4))

In [ ]:
# # ------------------------------------------------------------
# # Time-binned vs raw-flat comparison
# # ------------------------------------------------------------
#
# main_comp = cross_model_df[[
#     "dataset",
#     "task",
#     "model",
#     "main_mean_auc",
# ]].copy()
#
# main_comp = main_comp.rename(columns={
#     "main_mean_auc": "rawflat_0_0p6_auc",
# })
#
# timebin_comp = refinement_summary_df[
#     refinement_summary_df["group"] == "timebin_0_0p6"
#     ][[
#     "dataset",
#     "task",
#     "model",
#     "mean_subject_auc",
# ]].copy()
#
# timebin_comp = timebin_comp.rename(columns={
#     "mean_subject_auc": "timebin_0_0p6_auc",
# })
#
# timebin_vs_rawflat_df = main_comp.merge(
#     timebin_comp,
#     on=["dataset", "task", "model"],
#     how="outer",
# )
#
# timebin_vs_rawflat_df["delta_timebin_minus_rawflat"] = (
#         timebin_vs_rawflat_df["timebin_0_0p6_auc"]
#         - timebin_vs_rawflat_df["rawflat_0_0p6_auc"]
# )
#
# timebin_vs_rawflat_df = (
#     timebin_vs_rawflat_df
#     .sort_values(["dataset", "model"])
#     .reset_index(drop=True)
# )
#
# save_dataframe(
#     timebin_vs_rawflat_df,
#     TABLE_DIR / "timebin_vs_rawflat_summary.csv",
# )
#
# display(round_numeric_columns(timebin_vs_rawflat_df, 4))
#
#
# # ------------------------------------------------------------
# # Wide-window time-binned vs regular time-binned comparison
# # ------------------------------------------------------------
#
# wide_comp = refinement_summary_df[
#     refinement_summary_df["group"] == "wide_timebin_0_0p8"
#     ][[
#     "dataset",
#     "task",
#     "model",
#     "mean_subject_auc",
# ]].copy()
#
# wide_comp = wide_comp.rename(columns={
#     "mean_subject_auc": "wide_timebin_0_0p8_auc",
# })
#
# timebin_only = timebin_comp.copy()
#
# wide_vs_timebin_df = timebin_only.merge(
#     wide_comp,
#     on=["dataset", "task", "model"],
#     how="outer",
# )
#
# wide_vs_timebin_df["delta_wide_minus_timebin"] = (
#         wide_vs_timebin_df["wide_timebin_0_0p8_auc"]
#         - wide_vs_timebin_df["timebin_0_0p6_auc"]
# )
#
# wide_vs_timebin_df = (
#     wide_vs_timebin_df
#     .sort_values(["dataset", "model"])
#     .reset_index(drop=True)
# )
#
# save_dataframe(
#     wide_vs_timebin_df,
#     TABLE_DIR / "wide_timebin_vs_timebin_summary.csv",
# )
#
# display(round_numeric_columns(wide_vs_timebin_df, 4))
#
#
# # ------------------------------------------------------------
# # ds006480 active/passive condition comparison
# # ------------------------------------------------------------
#
# ds006480_condition_df = refinement_summary_df[
#     refinement_summary_df["group"].isin([
#         "ds006480_condition",
#         "ds006480_extra_condition",
#     ])
# ].copy()
#
# save_dataframe(
#     ds006480_condition_df,
#     TABLE_DIR / "ds006480_condition_model_summary.csv",
# )
#
# display(round_numeric_columns(ds006480_condition_df, 4))

# ------------------------------------------------------------
# Selected high-subject-count configuration summary
# ------------------------------------------------------------

wanted_cols = [
    "dataset",
    "task",
    "model",
    "main_mean_auc",
    "main_std_auc",
    "loso_mean_auc",
    "loso_std_auc",
]

available_cols = [c for c in wanted_cols if c in cross_model_df.columns]

selected_config_summary_df = cross_model_df[available_cols].copy()

rename_map = {
    "main_mean_auc": "within_subject_mean_auc",
    "main_std_auc": "within_subject_std_auc",
    "loso_mean_auc": "loso_mean_auc",
    "loso_std_auc": "loso_std_auc",
}

selected_config_summary_df = selected_config_summary_df.rename(
    columns={k: v for k, v in rename_map.items() if k in selected_config_summary_df.columns}
)

selected_config_summary_df["selected_config"] = "max30_wide_timebin"
selected_config_summary_df["classical_feature_representation"] = "50ms_time_bins_0_0_to_0_8s"
selected_config_summary_df["eegnet_input_representation"] = "raw_epochs_0_0_to_0_8s"

selected_config_summary_df = (
    selected_config_summary_df
    .sort_values(["dataset", "model"])
    .reset_index(drop=True)
)

save_dataframe(
    selected_config_summary_df,
    TABLE_DIR / "selected_max30_wide_timebin_model_summary.csv",
)

display(round_numeric_columns(selected_config_summary_df, 4))

In [ ]:
if len(refinement_summary_df) > 0:
    # Timebin vs raw-flat plot
    plot_df = timebin_vs_rawflat_df.copy()
    plot_df = plot_df[plot_df["model"].isin(MODEL_ORDER)]

    fig, ax = plt.subplots(figsize=(10, 5))

    x_labels = [
        f"{r.dataset}\n{model_display_name(r.model)}"
        for _, r in plot_df.iterrows()
    ]
    x = np.arange(len(plot_df))
    w = 0.38

    ax.bar(
        x - w / 2,
        plot_df["rawflat_0_0p6_auc"].astype(float),
        width=w,
        label="Raw-flat 0.0–0.6 s",
    )

    ax.bar(
        x + w / 2,
        plot_df["timebin_0_0p6_auc"].astype(float),
        width=w,
        label="Time-binned 0.0–0.6 s",
    )

    ax.axhline(0.5, linestyle="--", linewidth=1)
    ax.set_xticks(x)
    ax.set_xticklabels(x_labels, rotation=45, ha="right")
    ax.set_ylim(0.0, 1.0)
    ax.set_ylabel("Mean subject ROC-AUC")
    ax.set_title("Raw-flat vs time-binned decoding")
    ax.grid(axis="y", alpha=0.3)
    ax.legend()

    fig.tight_layout()
    save_figure(fig, FIG_DIR / "rawflat_vs_timebin_auc.png")
    plt.show()


    # Wide timebin vs normal timebin plot
    plot_df = wide_vs_timebin_df.copy()
    plot_df = plot_df[plot_df["model"].isin(MODEL_ORDER)]

    fig, ax = plt.subplots(figsize=(10, 5))

    x_labels = [
        f"{r.dataset}\n{model_display_name(r.model)}"
        for _, r in plot_df.iterrows()
    ]
    x = np.arange(len(plot_df))
    w = 0.38

    ax.bar(
        x - w / 2,
        plot_df["timebin_0_0p6_auc"].astype(float),
        width=w,
        label="Time-binned 0.0–0.6 s",
    )

    ax.bar(
        x + w / 2,
        plot_df["wide_timebin_0_0p8_auc"].astype(float),
        width=w,
        label="Time-binned 0.0–0.8 s",
    )

    ax.axhline(0.5, linestyle="--", linewidth=1)
    ax.set_xticks(x)
    ax.set_xticklabels(x_labels, rotation=45, ha="right")
    ax.set_ylim(0.0, 1.0)
    ax.set_ylabel("Mean subject ROC-AUC")
    ax.set_title("Regular vs wide post-stimulus time-binned decoding")
    ax.grid(axis="y", alpha=0.3)
    ax.legend()

    fig.tight_layout()
    save_figure(fig, FIG_DIR / "timebin_vs_wide_timebin_auc.png")
    plt.show()


    # ds006480 active/passive condition plot
    if len(ds006480_condition_df) > 0:
        plot_df = ds006480_condition_df.copy()
        plot_df = plot_df[plot_df["model"].isin(MODEL_ORDER)]

        fig, ax = plt.subplots(figsize=(10, 5))

        x_labels = [
            f"{r.event_preset}\n{r.feature_mode}\n{model_display_name(r.model)}"
            for _, r in plot_df.iterrows()
        ]
        x = np.arange(len(plot_df))

        ax.bar(
            x,
            plot_df["mean_subject_auc"].astype(float),
        )

        ax.axhline(0.5, linestyle="--", linewidth=1)
        ax.set_xticks(x)
        ax.set_xticklabels(x_labels, rotation=45, ha="right")
        ax.set_ylim(0.0, 1.0)
        ax.set_ylabel("Mean subject ROC-AUC")
        ax.set_title("ds006480 active/passive condition-separated decoding")
        ax.grid(axis="y", alpha=0.3)

        fig.tight_layout()
        save_figure(fig, FIG_DIR / "ds006480_condition_decoding_auc.png")
        plt.show()

In [ ]:
def load_model_summary_for_experiment(exp_key: str, group_name: str) -> pd.DataFrame:
    """
    Load model_summary_across_subjects.csv for one standalone experiment key.
    """
    cfg = get_experiment(exp_key)
    df = safe_read_result_csv(cfg, "model_summary_across_subjects.csv")

    if df is None:
        print(f"[MISSING] {group_name}: {exp_key}")
        print("Expected under:", cfg.result_dir())
        return pd.DataFrame()

    if "model" not in df.columns or "mean_subject_auc" not in df.columns:
        print(f"[WARN] {group_name}: {exp_key} missing required columns.")
        print("Columns:", df.columns.tolist())
        print("Source :", get_source_path(df))
        return pd.DataFrame()

    out_rows = []

    for _, r in df.iterrows():
        model = normalize_model_name(r["model"])

        out_rows.append({
            "group": group_name,
            "experiment_key": exp_key,
            "dataset": cfg.dataset.dataset,
            "task": cfg.dataset.task,
            "event_preset": cfg.dataset.event_preset,
            "run_tag": cfg.run_tag,
            "crop_window": str(cfg.effective_crop_window()),
            "feature_mode": cfg.feature.mode,
            "feature_bin_ms": cfg.feature.bin_ms,
            "model": model,
            "model_display": model_display_name(model),
            "mean_subject_auc": safe_float(r.get("mean_subject_auc")),
            "std_subject_auc": safe_float(r.get("std_subject_auc")),
            "median_subject_auc": safe_float(r.get("median_subject_auc")),
            "worst_subject_auc": safe_float(r.get("worst_subject_auc")),
            "best_subject_auc": safe_float(r.get("best_subject_auc")),
            "mean_subject_pr_auc": safe_float(r.get("mean_subject_pr_auc")),
            "mean_subject_bal_acc": safe_float(r.get("mean_subject_bal_acc")),
            "mean_subject_acc": safe_float(r.get("mean_subject_acc")),
            "mean_subject_f1": safe_float(r.get("mean_subject_f1")),
            "n_subjects": safe_float(r.get("n_subjects")),
            "source_path": get_source_path(df),
        })

    return pd.DataFrame(out_rows)


refinement_frames = []

for key in TIMEBIN_EXPERIMENT_KEYS:
    refinement_frames.append(load_model_summary_for_experiment(key, "timebin_0_0p6"))

for key in WIDE_TIMEBIN_EXPERIMENT_KEYS:
    refinement_frames.append(load_model_summary_for_experiment(key, "wide_timebin_0_0p8"))

for key in DS006480_CONDITION_EXPERIMENT_KEYS:
    refinement_frames.append(load_model_summary_for_experiment(key, "ds006480_condition"))

for key in DS006480_EXTRA_CONDITION_EXPERIMENT_KEYS:
    refinement_frames.append(load_model_summary_for_experiment(key, "ds006480_extra_condition"))

refinement_frames = [df for df in refinement_frames if df is not None and len(df) > 0]

if len(refinement_frames) == 0:
    print("[WARN] No refinement summaries loaded.")
    refinement_summary_df = pd.DataFrame()
else:
    refinement_summary_df = (
        pd.concat(refinement_frames, ignore_index=True)
        .sort_values(["group", "dataset", "event_preset", "model"])
        .reset_index(drop=True)
    )

save_dataframe(
    refinement_summary_df,
    TABLE_DIR / "refinement_model_summary_all.csv",
)

display(round_numeric_columns(refinement_summary_df, 4))

In [ ]:
rank_rows = []

for dataset, sub_df in cross_model_df.groupby("dataset"):
    sub_df = sub_df.sort_values(
        ["main_mean_auc", "main_median_auc", "main_worst_auc"],
        ascending=False,
    ).reset_index(drop=True)

    for rank, (_, r) in enumerate(sub_df.iterrows(), start=1):
        out = r.to_dict()
        out["main_rank"] = int(rank)
        rank_rows.append(out)

cross_model_ranked_df = pd.DataFrame(rank_rows)

save_dataframe(cross_model_ranked_df, TABLE_DIR / "cross_dataset_model_ranked.csv")

display(round_numeric_columns(cross_model_ranked_df, 4))

In [ ]:
def get_model_row(df: pd.DataFrame | None, model: str) -> pd.Series | None:
    if df is None or "model" not in df.columns:
        return None

    tmp = df.copy()
    tmp["model"] = tmp["model"].map(normalize_model_name)

    rows = tmp[tmp["model"] == normalize_model_name(model)]

    if len(rows) == 0:
        return None

    return rows.iloc[0]


lda_rows = []

for dataset, obj in loaded.items():
    main_model_df = obj["main_model"]
    perm_model_df = obj["perm_model"]
    prestim_model_df = obj["prestim_model"]
    loso_model_df = obj["loso_model"]

    main_lda = get_model_row(main_model_df, PRIMARY_MODEL)
    perm_lda = get_model_row(perm_model_df, PRIMARY_MODEL)
    prestim_lda = get_model_row(prestim_model_df, PRIMARY_MODEL)
    loso_lda = get_model_row(loso_model_df, PRIMARY_MODEL)

    ranked_sub = cross_model_ranked_df[
        (cross_model_ranked_df["dataset"] == dataset) &
        (cross_model_ranked_df["model"] == PRIMARY_MODEL)
        ]

    if len(ranked_sub) > 0:
        lda_rank = int(ranked_sub.iloc[0]["main_rank"])
    else:
        lda_rank = np.nan

    if main_lda is not None:
        main_auc = safe_float(main_lda.get("mean_subject_auc"))
        main_worst_auc = safe_float(main_lda.get("worst_subject_auc"))
        main_n_subjects = safe_float(main_lda.get("n_subjects"))
    else:
        main_auc = np.nan
        main_worst_auc = np.nan
        main_n_subjects = np.nan

    perm_auc = safe_float(perm_lda.get("mean_subject_auc")) if perm_lda is not None else np.nan
    prestim_auc = safe_float(prestim_lda.get("mean_subject_auc")) if prestim_lda is not None else np.nan
    loso_auc = safe_float(loso_lda.get("mean_subject_auc")) if loso_lda is not None else np.nan
    loso_worst_auc = safe_float(loso_lda.get("worst_subject_auc")) if loso_lda is not None else np.nan

    lda_rows.append({
        "dataset": dataset,
        "task": obj["task"],
        "signal": obj["signal"],
        "n_subjects_main": main_n_subjects,
        "lda_main_rank": lda_rank,
        "lda_main_mean_auc": main_auc,
        "lda_main_worst_auc": main_worst_auc,
        "lda_permuted_mean_auc": perm_auc,
        "lda_prestim_mean_auc": prestim_auc,
        "lda_loso_mean_auc": loso_auc,
        "lda_loso_worst_auc": loso_worst_auc,
        "delta_main_minus_permuted": main_auc - perm_auc if np.isfinite(main_auc) and np.isfinite(perm_auc) else np.nan,
        "delta_main_minus_prestim": main_auc - prestim_auc if np.isfinite(main_auc) and np.isfinite(prestim_auc) else np.nan,
        "delta_main_minus_loso": main_auc - loso_auc if np.isfinite(main_auc) and np.isfinite(loso_auc) else np.nan,
        "has_main": main_lda is not None,
        "has_permuted": perm_lda is not None,
        "has_prestim": prestim_lda is not None,
        "has_loso": loso_lda is not None,
    })

lda_replication_df = (
    pd.DataFrame(lda_rows)
    .sort_values("dataset")
    .reset_index(drop=True)
)

save_dataframe(lda_replication_df, TABLE_DIR / "lda_cross_dataset_replication_table.csv")

display(round_numeric_columns(lda_replication_df, 4))

In [ ]:
def label_bool(x: bool) -> str:
    return "yes" if bool(x) else "no"


status_rows = []

for _, r in lda_replication_df.iterrows():
    dataset = r["dataset"]

    has_main = bool(r["has_main"])
    has_controls = bool(r["has_permuted"]) and bool(r["has_prestim"])
    has_loso = bool(r["has_loso"])

    main_auc = safe_float(r["lda_main_mean_auc"])
    rank = safe_float(r["lda_main_rank"])
    d_perm = safe_float(r["delta_main_minus_permuted"])
    d_prestim = safe_float(r["delta_main_minus_prestim"])

    above_chance = np.isfinite(main_auc) and main_auc > CHANCE_AUC
    above_success_threshold = np.isfinite(main_auc) and main_auc >= MIN_MAIN_AUC_FOR_SUCCESS
    top_k = np.isfinite(rank) and rank <= TOP_K_ACCEPTED

    permuted_drop_ok = np.isfinite(d_perm) and d_perm >= MIN_MAIN_MINUS_PERMUTED_DELTA
    prestim_drop_ok = np.isfinite(d_prestim) and d_prestim >= MIN_MAIN_MINUS_PRESTIM_DELTA

    controls_ok = has_controls and permuted_drop_ok and prestim_drop_ok

    if has_main and above_success_threshold and top_k and controls_ok:
        replication_status = "strong_replication"
    elif has_main and above_chance and top_k:
        replication_status = "partial_replication"
    elif has_main:
        replication_status = "weak_or_mixed"
    else:
        replication_status = "missing_main_results"

    status_rows.append({
        "dataset": dataset,
        "task": r["task"],
        "is_reference_dataset": dataset == REFERENCE_DATASET,
        "has_main_results": has_main,
        "has_controls": has_controls,
        "has_loso": has_loso,
        "lda_main_mean_auc": main_auc,
        "lda_main_rank": rank,
        "lda_above_chance": above_chance,
        "lda_above_success_threshold": above_success_threshold,
        "lda_top_k": top_k,
        "permuted_drop_ok": permuted_drop_ok,
        "prestim_drop_ok": prestim_drop_ok,
        "controls_ok": controls_ok,
        "replication_status": replication_status,
    })

replication_status_df = (
    pd.DataFrame(status_rows)
    .sort_values(["is_reference_dataset", "dataset"], ascending=[False, True])
    .reset_index(drop=True)
)

save_dataframe(replication_status_df, TABLE_DIR / "replication_status_by_dataset.csv")

display(round_numeric_columns(replication_status_df, 4))

In [ ]:
completed_main_datasets = replication_status_df[
    replication_status_df["has_main_results"] == True
    ]["dataset"].tolist()

additional_completed = [
    d for d in completed_main_datasets
    if d != REFERENCE_DATASET
]

strong_or_partial_additional = replication_status_df[
    (replication_status_df["dataset"] != REFERENCE_DATASET) &
    (replication_status_df["replication_status"].isin(["strong_replication", "partial_replication"]))
    ]["dataset"].tolist()

n_additional_completed = len(additional_completed)
n_additional_replicated = len(strong_or_partial_additional)

replication_requirement_met_by_outputs = n_additional_completed >= REQUIRED_ADDITIONAL_DATASETS
replication_requirement_met_by_replication_pattern = n_additional_replicated >= REQUIRED_ADDITIONAL_DATASETS

checklist = {
    "reference_dataset": REFERENCE_DATASET,
    "required_additional_datasets": REQUIRED_ADDITIONAL_DATASETS,
    "datasets_with_main_outputs": completed_main_datasets,
    "additional_datasets_with_main_outputs": additional_completed,
    "additional_datasets_with_strong_or_partial_replication": strong_or_partial_additional,
    "n_additional_completed": n_additional_completed,
    "n_additional_replicated": n_additional_replicated,
    "requirement_met_by_completed_outputs": bool(replication_requirement_met_by_outputs),
    "requirement_met_by_replication_pattern": bool(replication_requirement_met_by_replication_pattern),
}

save_json(checklist, TABLE_DIR / "replication_checklist.json")

print("=" * 90)
print("Replication checklist")
print("=" * 90)

print("Reference dataset:", checklist["reference_dataset"])
print("Required additional datasets:", checklist["required_additional_datasets"])
print("Datasets with main outputs:", checklist["datasets_with_main_outputs"])
print("Additional completed datasets:", checklist["additional_datasets_with_main_outputs"])
print("Additional replicated datasets:", checklist["additional_datasets_with_strong_or_partial_replication"])
print()
print("Requirement met by completed outputs:", checklist["requirement_met_by_completed_outputs"])
print("Requirement met by replication pattern:", checklist["requirement_met_by_replication_pattern"])

if not replication_requirement_met_by_outputs:
    missing_n = REQUIRED_ADDITIONAL_DATASETS - n_additional_completed
    print()
    print(f"Status: you still need benchmark outputs for {missing_n} additional dataset(s).")
elif not replication_requirement_met_by_replication_pattern:
    print()
    print("Status: enough additional datasets have outputs, but the replication pattern is mixed.")
else:
    print()
    print("Status: the saved outputs support reporting that the result was tested on two additional datasets.")

In [ ]:
heat_df = (
    cross_model_df
    .pivot_table(
        index="dataset",
        columns="model",
        values="main_mean_auc",
        aggfunc="mean",
    )
)

for model in MODEL_ORDER:
    if model not in heat_df.columns:
        heat_df[model] = np.nan

heat_df = heat_df[MODEL_ORDER]
heat_df = heat_df.sort_index()

fig, ax = plt.subplots(figsize=(8.5, max(3.8, 0.6 * len(heat_df) + 1.5)))

im = ax.imshow(heat_df.values.astype(float), aspect="auto", vmin=0.0, vmax=1.0)

ax.set_xticks(np.arange(len(heat_df.columns)))
ax.set_yticks(np.arange(len(heat_df.index)))
ax.set_xticklabels([model_display_name(m) for m in heat_df.columns])
ax.set_yticklabels(heat_df.index)

ax.set_title("Cross-dataset grouped within-subject ROC-AUC")
ax.set_xlabel("Model")
ax.set_ylabel("Dataset")

for i in range(heat_df.shape[0]):
    for j in range(heat_df.shape[1]):
        val = heat_df.iloc[i, j]
        text = "" if pd.isna(val) else f"{val:.3f}"
        ax.text(j, i, text, ha="center", va="center")

cbar = fig.colorbar(im, ax=ax)
cbar.set_label("Mean subject ROC-AUC")

fig.tight_layout()

save_figure(fig, FIG_DIR / "cross_dataset_model_auc_heatmap.png")
plt.show()

In [ ]:
plot_df = lda_replication_df.copy()
plot_df = plot_df.sort_values("dataset").reset_index(drop=True)

fig, ax = plt.subplots(figsize=(9, 4.8))

x = np.arange(len(plot_df))

ax.bar(
    x,
    plot_df["lda_main_mean_auc"].astype(float),
)

ax.axhline(CHANCE_AUC, linestyle="--", linewidth=1, label="Chance AUC = 0.50")
ax.axhline(MIN_MAIN_AUC_FOR_SUCCESS, linestyle=":", linewidth=1, label=f"Success threshold = {MIN_MAIN_AUC_FOR_SUCCESS:.2f}")

ax.set_xticks(x)
ax.set_xticklabels(plot_df["dataset"])
ax.set_ylim(0.0, 1.0)
ax.set_ylabel("LDA mean subject ROC-AUC")
ax.set_xlabel("Dataset")
ax.set_title("LDA replication across datasets")
ax.grid(axis="y", alpha=0.3)
ax.legend()

for i, v in enumerate(plot_df["lda_main_mean_auc"].astype(float)):
    if np.isfinite(v):
        ax.text(i, v + 0.02, f"{v:.3f}", ha="center")

fig.tight_layout()

save_figure(fig, FIG_DIR / "lda_replication_across_datasets.png")
plt.show()

In [ ]:
control_plot_df = lda_replication_df.copy()
control_plot_df = control_plot_df.sort_values("dataset").reset_index(drop=True)

x = np.arange(len(control_plot_df))
w = 0.25

fig, ax = plt.subplots(figsize=(10, 5))

ax.bar(
    x - w,
    control_plot_df["lda_main_mean_auc"].astype(float),
    width=w,
    label="Main benchmark",
)

ax.bar(
    x,
    control_plot_df["lda_permuted_mean_auc"].astype(float),
    width=w,
    label="Permuted labels",
)

ax.bar(
    x + w,
    control_plot_df["lda_prestim_mean_auc"].astype(float),
    width=w,
    label="Pre-stimulus-only",
)

ax.axhline(CHANCE_AUC, linestyle="--", linewidth=1, label="Chance AUC = 0.50")

ax.set_xticks(x)
ax.set_xticklabels(control_plot_df["dataset"])
ax.set_ylim(0.0, 1.0)
ax.set_ylabel("Mean subject ROC-AUC")
ax.set_xlabel("Dataset")
ax.set_title("LDA main benchmark vs controls across datasets")
ax.grid(axis="y", alpha=0.3)
ax.legend()

fig.tight_layout()

save_figure(fig, FIG_DIR / "lda_main_vs_controls_across_datasets.png")
plt.show()

In [ ]:
loso_plot_df = lda_replication_df.copy()
loso_plot_df = loso_plot_df.sort_values("dataset").reset_index(drop=True)

x = np.arange(len(loso_plot_df))
w = 0.35

fig, ax = plt.subplots(figsize=(9, 4.8))

ax.bar(
    x - w / 2,
    loso_plot_df["lda_main_mean_auc"].astype(float),
    width=w,
    label="Grouped within-subject",
)

ax.bar(
    x + w / 2,
    loso_plot_df["lda_loso_mean_auc"].astype(float),
    width=w,
    label="LOSO",
)

ax.axhline(CHANCE_AUC, linestyle="--", linewidth=1, label="Chance AUC = 0.50")

ax.set_xticks(x)
ax.set_xticklabels(loso_plot_df["dataset"])
ax.set_ylim(0.0, 1.0)
ax.set_ylabel("Mean subject ROC-AUC")
ax.set_xlabel("Dataset")
ax.set_title("LDA within-subject vs LOSO across datasets")
ax.grid(axis="y", alpha=0.3)
ax.legend()

fig.tight_layout()

save_figure(fig, FIG_DIR / "lda_within_subject_vs_loso_across_datasets.png")
plt.show()

In [ ]:
rank_pivot = (
    cross_model_ranked_df
    .pivot_table(
        index="dataset",
        columns="model",
        values="main_rank",
        aggfunc="first",
    )
)

for model in MODEL_ORDER:
    if model not in rank_pivot.columns:
        rank_pivot[model] = np.nan

rank_pivot = rank_pivot[MODEL_ORDER].sort_index()

fig, ax = plt.subplots(figsize=(8.5, max(3.8, 0.6 * len(rank_pivot) + 1.5)))

im = ax.imshow(rank_pivot.values.astype(float), aspect="auto", vmin=1, vmax=len(MODEL_ORDER))

ax.set_xticks(np.arange(len(rank_pivot.columns)))
ax.set_yticks(np.arange(len(rank_pivot.index)))
ax.set_xticklabels([model_display_name(m) for m in rank_pivot.columns])
ax.set_yticklabels(rank_pivot.index)

ax.set_title("Model rank by dataset")
ax.set_xlabel("Model")
ax.set_ylabel("Dataset")

for i in range(rank_pivot.shape[0]):
    for j in range(rank_pivot.shape[1]):
        val = rank_pivot.iloc[i, j]
        text = "" if pd.isna(val) else f"{int(val)}"
        ax.text(j, i, text, ha="center", va="center")

cbar = fig.colorbar(im, ax=ax)
cbar.set_label("Rank; lower is better")

fig.tight_layout()

save_figure(fig, FIG_DIR / "cross_dataset_model_rank_heatmap.png")
plt.show()

display(rank_pivot)

In [ ]:
best_model_rows = []

for_dataset_groups = cross_model_ranked_df.groupby("dataset")

for dataset, sub_df in for_dataset_groups:
    best = sub_df.sort_values("main_rank").iloc[0]
    lda = sub_df[sub_df["model"] == PRIMARY_MODEL]

    if len(lda) > 0:
        lda = lda.iloc[0]
        lda_auc = safe_float(lda["main_mean_auc"])
        lda_rank = safe_float(lda["main_rank"])
    else:
        lda_auc = np.nan
        lda_rank = np.nan

    best_auc = safe_float(best["main_mean_auc"])

    best_model_rows.append({
        "dataset": dataset,
        "best_model": best["model"],
        "best_model_display": model_display_name(best["model"]),
        "best_model_auc": best_auc,
        "lda_auc": lda_auc,
        "lda_rank": lda_rank,
        "delta_best_minus_lda": best_auc - lda_auc if np.isfinite(best_auc) and np.isfinite(lda_auc) else np.nan,
        "lda_is_best": normalize_model_name(best["model"]) == PRIMARY_MODEL,
        "lda_is_top_2": np.isfinite(lda_rank) and lda_rank <= 2,
    })

best_model_df = (
    pd.DataFrame(best_model_rows)
    .sort_values("dataset")
    .reset_index(drop=True)
)

save_dataframe(best_model_df, TABLE_DIR / "dataset_best_model_vs_lda.csv")

display(round_numeric_columns(best_model_df, 4))

In [ ]:
best_model_rows = []

for dataset, sub_df in cross_model_ranked_df.groupby("dataset"):
    best = sub_df.sort_values("main_rank").iloc[0]
    lda = sub_df[sub_df["model"] == PRIMARY_MODEL]

    if len(lda) > 0:
        lda = lda.iloc[0]
        lda_auc = safe_float(lda["main_mean_auc"])
        lda_rank = safe_float(lda["main_rank"])
    else:
        lda_auc = np.nan
        lda_rank = np.nan

    best_auc = safe_float(best["main_mean_auc"])

    best_model_rows.append({
        "dataset": dataset,
        "best_model": best["model"],
        "best_model_display": model_display_name(best["model"]),
        "best_model_auc": best_auc,
        "lda_auc": lda_auc,
        "lda_rank": lda_rank,
        "delta_best_minus_lda": best_auc - lda_auc if np.isfinite(best_auc) and np.isfinite(lda_auc) else np.nan,
        "lda_is_best": normalize_model_name(best["model"]) == PRIMARY_MODEL,
        "lda_is_top_2": np.isfinite(lda_rank) and lda_rank <= 2,
    })

best_model_df = (
    pd.DataFrame(best_model_rows)
    .sort_values("dataset")
    .reset_index(drop=True)
)

save_dataframe(best_model_df, TABLE_DIR / "dataset_best_model_vs_lda.csv")

display(round_numeric_columns(best_model_df, 4))

In [ ]:
thesis_table = lda_replication_df[[
    "dataset",
    "task",
    "n_subjects_main",
    "lda_main_rank",
    "lda_main_mean_auc",
    "lda_main_worst_auc",
    "lda_permuted_mean_auc",
    "lda_prestim_mean_auc",
    "lda_loso_mean_auc",
    "delta_main_minus_permuted",
    "delta_main_minus_prestim",
]].copy()

thesis_table = thesis_table.merge(
    replication_status_df[[
        "dataset",
        "replication_status",
        "controls_ok",
    ]],
    on="dataset",
    how="left",
)

thesis_table = thesis_table.sort_values("dataset").reset_index(drop=True)

save_dataframe(thesis_table, TABLE_DIR / "thesis_ready_cross_dataset_replication_summary.csv")

display(round_numeric_columns(thesis_table, 4))

In [ ]:
def save_table_latex(
        df: pd.DataFrame,
        path: Path,
        decimals: int = 4,
        index: bool = False,
):
    out = round_numeric_columns(df.copy(), decimals=decimals)

    rename_map = {
        "dataset": "Dataset",
        "task": "Task",
        "n_subjects_main": "N",
        "lda_main_rank": "LDA Rank",
        "lda_main_mean_auc": "LDA Main AUC",
        "lda_main_worst_auc": "LDA Worst AUC",
        "lda_permuted_mean_auc": "Permuted AUC",
        "lda_prestim_mean_auc": "Pre-stim AUC",
        "lda_loso_mean_auc": "LOSO AUC",
        "delta_main_minus_permuted": "$\\Delta$ Main--Permuted",
        "delta_main_minus_prestim": "$\\Delta$ Main--Pre-stim",
        "replication_status": "Status",
        "controls_ok": "Controls OK",
    }

    out = out.rename(columns=rename_map)

    latex = out.to_latex(index=index, escape=False)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(latex, encoding="utf-8")


save_table_latex(
    thesis_table,
    TABLE_DIR / "thesis_ready_cross_dataset_replication_summary.tex",
    decimals=4,
    index=False,
)

print("Saved LaTeX table:")
print(TABLE_DIR / "thesis_ready_cross_dataset_replication_summary.tex")

In [ ]:
n_total_datasets_with_main = int(replication_status_df["has_main_results"].sum())
n_additional_completed = len(additional_completed)
n_additional_replicated = len(strong_or_partial_additional)

print("=" * 90)
print("Draft interpretation")
print("=" * 90)

if replication_requirement_met_by_replication_pattern:
    interpretation = f"""
The pipeline was evaluated on the original reference dataset ({REFERENCE_DATASET}) and on two additional OpenNeuro EEG/ERP datasets. Across the completed dataset set, LDA remained a strong retained baseline under the grouped within-subject benchmark. The cross-dataset comparison also confirmed that the main benchmark generally exceeded the anti-overfitting controls, supporting the interpretation that the classifier was using task-related post-stimulus information rather than only trivial structure. These replication results strengthen the dissertation by showing that the main conclusion is not limited to a single OpenNeuro dataset.
"""
elif replication_requirement_met_by_outputs:
    interpretation = f"""
The pipeline has now been run on the original reference dataset ({REFERENCE_DATASET}) and on the required number of additional datasets. However, the replication pattern is mixed under the conservative status rule used in this notebook. This means the dissertation can report that the pipeline was tested across datasets, but the written interpretation should be cautious: the results support broader evaluation of the pipeline, but they do not show identical model behavior across every dataset.
"""
else:
    missing_n = REQUIRED_ADDITIONAL_DATASETS - n_additional_completed

    interpretation = f"""
At this stage, the saved outputs do not yet fully satisfy the replication requirement. The current comparison includes {n_additional_completed} additional dataset(s) beyond the reference dataset ({REFERENCE_DATASET}), but {REQUIRED_ADDITIONAL_DATASETS} additional datasets are required. One more suitable OpenNeuro EEG/ERP dataset still needs to be processed through the same benchmark pipeline before the dissertation can claim that the result was replicated on two additional datasets.
"""

print(interpretation.strip())

draft_path = TABLE_DIR / "draft_cross_dataset_replication_interpretation.txt"
draft_path.write_text(interpretation.strip(), encoding="utf-8")

print()
print("Saved draft text to:", draft_path)

In [ ]:
artifact_manifest = {
    "output_dir": str(OUT_DIR),
    "table_dir": str(TABLE_DIR),
    "figure_dir": str(FIG_DIR),
    "tables": sorted([p.name for p in TABLE_DIR.glob("*") if p.is_file()]),
    "figures": sorted([p.name for p in FIG_DIR.glob("*") if p.is_file()]),
    "checklist": checklist,
}

save_json(
    artifact_manifest,
    OUT_DIR / "cross_dataset_replication_manifest.json",
)

print("Saved cross-dataset replication artifacts.")
print()
print("Tables:")
for p in sorted(TABLE_DIR.glob("*")):
    if p.is_file():
        print(" -", p.name)

print()
print("Figures:")
for p in sorted(FIG_DIR.glob("*")):
    if p.is_file():
        print(" -", p.name)

print()
print("Manifest:")
print(OUT_DIR / "cross_dataset_replication_manifest.json")

In [ ]:
# ------------------------------------------------------------
# Final selected high-count within-subject + LOSO summary
# ------------------------------------------------------------

selected_within_path = TABLE_DIR / "selected_max30_wide_timebin_model_summary.csv"

loso_summary_path = (
    PROJECT_ROOT
    / "results"
    / "thesis_artifacts"
    / "loso_susceptibility_fdr"
    / "tables"
    / "loso_model_summary_from_subject_auc.csv"
)

selected_within_df = pd.read_csv(selected_within_path)
loso_summary_df = pd.read_csv(loso_summary_path)

selected_within_df["dataset_label"] = selected_within_df["dataset"].astype(str)
selected_within_df.loc[
    selected_within_df["dataset"] == "ds006480",
    "dataset_label"
] = "ds006480_pooled"

final_within_loso_summary_df = selected_within_df.merge(
    loso_summary_df[[
        "dataset_label",
        "model",
        "n_subjects",
        "loso_mean_auc",
        "loso_std_auc",
        "loso_median_auc",
        "loso_min_auc",
        "loso_max_auc",
    ]],
    on=["dataset_label", "model"],
    how="left",
)

final_within_loso_summary_df = final_within_loso_summary_df[[
    "dataset",
    "dataset_label",
    "task",
    "model",
    "n_subjects",
    "within_subject_mean_auc",
    "within_subject_std_auc",
    "loso_mean_auc",
    "loso_std_auc",
    "loso_median_auc",
    "loso_min_auc",
    "loso_max_auc",
    "selected_config",
    "classical_feature_representation",
    "eegnet_input_representation",
]]

final_within_loso_summary_df = (
    final_within_loso_summary_df
    .sort_values(["dataset", "model"])
    .reset_index(drop=True)
)

save_dataframe(
    final_within_loso_summary_df,
    TABLE_DIR / "final_max30_wide_timebin_within_loso_model_summary.csv",
)

display(round_numeric_columns(final_within_loso_summary_df, 4))